# Modelo de Regresión Logística — Proyecto Minería de Datos

Este notebook carga directamente los archivos `.sav` del repositorio, construye una variable respuesta binaria llamada `segundo_matrimonio`, prepara los datos, entrena varias configuraciones de regresión logística y evalúa los resultados con matriz de confusión y métricas de clasificación.

In [ ]:

import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 120)

In [ ]:
# ============================================================
# 2. Carga de datos .sav
# ============================================================

def leer_sav_seguro(ruta):
    """Lee un archivo .sav y agrega el nombre del archivo como referencia."""
    df = pd.read_spss(ruta)
    df["archivo_origen"] = os.path.basename(ruta)
    return df

rutas_matrimonios = sorted(glob.glob("data_matrimonios/*.sav"))
rutas_divorcios = sorted(glob.glob("data_divorcios/*.sav"))

print("Archivos de matrimonios encontrados:", len(rutas_matrimonios))
print("Archivos de divorcios encontrados:", len(rutas_divorcios))

# Para regresión logística usaremos principalmente matrimonios, porque allí se puede construir
# la variable respuesta relacionada con si una persona ya estuvo casada anteriormente.
matrimonios = pd.concat([leer_sav_seguro(ruta) for ruta in rutas_matrimonios], ignore_index=True)

print("Dimensiones de matrimonios:", matrimonios.shape)
display(matrimonios.head())
display(pd.DataFrame({"columna": matrimonios.columns, "tipo": matrimonios.dtypes.astype(str).values}).head(80))

In [ ]:
# ============================================================
# 3. Limpieza básica de nombres y exploración
# ============================================================

# Normalizamos nombres de columnas para evitar problemas con mayúsculas/espacios.
matrimonios.columns = (
    matrimonios.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

print("Columnas disponibles:")
print(matrimonios.columns.tolist())

print("\nTamaño del dataset:", matrimonios.shape)
print("Duplicados exactos:", matrimonios.duplicated().sum())

In [ ]:
# ============================================================
# 4. Construcción de variable respuesta: segundo_matrimonio
# ============================================================

# Buscamos automáticamente columnas que puedan indicar número de unión, estado civil previo
# o variables relacionadas con matrimonio anterior. Esto hace que el notebook sea más robusto
# aunque los nombres exactos de columnas cambien entre años.
palabras_clave = [
    "NUM", "UNION", "UNI", "NUP", "NUPCIAS", "ESTCIV", "CIVIL",
    "ANT", "ANTERIOR", "PREV", "MATR", "CAS"
]

candidatas = [c for c in matrimonios.columns if any(p in c for p in palabras_clave)]
print("Columnas candidatas para construir la variable respuesta:")
print(candidatas)

def crear_segundo_matrimonio(df):
    df = df.copy()
    cols = df.columns.tolist()

    # Prioridad 1: columnas típicas de número de unión/nupcias del hombre y la mujer.
    posibles_num_union = [
        c for c in cols
        if any(x in c for x in ["NUNU", "NUMUNI", "NUM_UNI", "NUP", "NUPCIAS", "UNION"])
    ]

    # Excluimos columnas claramente no útiles.
    posibles_num_union = [c for c in posibles_num_union if c not in ["ARCHIVO_ORIGEN"]]

    print("Columnas usadas como posibles indicadores de número de unión:", posibles_num_union)

    if len(posibles_num_union) == 0:
        raise ValueError(
            "No se encontró una columna clara para crear segundo_matrimonio. "
            "Revisa las columnas candidatas impresas arriba y ajusta manualmente la lista posibles_num_union."
        )

    indicadores = []
    for c in posibles_num_union:
        serie = pd.to_numeric(df[c], errors="coerce")
        # Si el número de unión es mayor a 1, se interpreta como segundo matrimonio o matrimonio posterior.
        indicadores.append(serie > 1)

    indicador_final = np.logical_or.reduce(indicadores)
    df["SEGUNDO_MATRIMONIO"] = indicador_final.astype(int)
    return df, posibles_num_union

matrimonios, columnas_respuesta = crear_segundo_matrimonio(matrimonios)

print("Distribución de la variable respuesta:")
display(matrimonios["SEGUNDO_MATRIMONIO"].value_counts(dropna=False).rename_axis("SEGUNDO_MATRIMONIO").reset_index(name="conteo"))
print("Proporción:")
display(matrimonios["SEGUNDO_MATRIMONIO"].value_counts(normalize=True).rename_axis("SEGUNDO_MATRIMONIO").reset_index(name="proporcion"))

## Justificación de la variable respuesta

La variable `SEGUNDO_MATRIMONIO` es una variable categórica binaria. Toma el valor 1 cuando alguno de los registros asociados al matrimonio indica que la persona ya había tenido una unión o matrimonio previo, y toma el valor 0 cuando corresponde a una primera unión. Se eligió esta variable porque permite plantear un problema de clasificación supervisada: predecir si un registro corresponde a un segundo matrimonio o matrimonio posterior a partir de características demográficas y del evento registrado.

In [ ]:
# ============================================================
# 5. Selección de variables predictoras
# ============================================================

# Eliminamos la variable respuesta y las columnas utilizadas directamente para construirla,
# porque dejarlas produciría fuga de información.
columnas_eliminar = set(["SEGUNDO_MATRIMONIO", "ARCHIVO_ORIGEN"]) | set(columnas_respuesta)

# También eliminamos columnas con demasiados valores faltantes o cardinalidad excesiva.
umbral_nulos = 0.70
prop_nulos = matrimonios.isna().mean()
cols_muchos_nulos = set(prop_nulos[prop_nulos > umbral_nulos].index)

X = matrimonios.drop(columns=list(columnas_eliminar | cols_muchos_nulos), errors="ignore")
y = matrimonios["SEGUNDO_MATRIMONIO"]

# Eliminamos columnas constantes.
cols_constantes = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
X = X.drop(columns=cols_constantes, errors="ignore")

print("Columnas eliminadas por fuga de información:", sorted(columnas_eliminar))
print("Columnas eliminadas por muchos nulos:", len(cols_muchos_nulos))
print("Columnas eliminadas por constantes:", cols_constantes)
print("Dimensiones finales de X:", X.shape)

display(X.head())

In [ ]:
# ============================================================
# 6. Separación entrenamiento/prueba
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Tamaño entrenamiento:", X_train.shape)
print("Tamaño prueba:", X_test.shape)
print("Distribución y_train:")
print(y_train.value_counts(normalize=True))
print("Distribución y_test:")
print(y_test.value_counts(normalize=True))

In [ ]:
# ============================================================
# 7. Preprocesamiento
# ============================================================

columnas_numericas = X.select_dtypes(include=["number", "int64", "float64"]).columns.tolist()
columnas_categoricas = [c for c in X.columns if c not in columnas_numericas]

print("Variables numéricas:", len(columnas_numericas))
print("Variables categóricas:", len(columnas_categoricas))

preprocesamiento = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), columnas_numericas),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
        ]), columnas_categoricas)
    ],
    remainder="drop"
)

In [ ]:
# ============================================================
# 8. Entrenamiento de varios modelos de Regresión Logística
# ============================================================

configuraciones = [
    {"nombre": "LogReg_1_baseline", "C": 1.0, "penalty": "l2", "class_weight": None, "solver": "lbfgs"},
    {"nombre": "LogReg_2_balanceado", "C": 1.0, "penalty": "l2", "class_weight": "balanced", "solver": "lbfgs"},
    {"nombre": "LogReg_3_regularizado", "C": 0.1, "penalty": "l2", "class_weight": "balanced", "solver": "lbfgs"},
    {"nombre": "LogReg_4_menos_regularizado", "C": 10.0, "penalty": "l2", "class_weight": "balanced", "solver": "lbfgs"},
]

resultados = []
modelos = {}

for cfg in configuraciones:
    modelo = Pipeline(steps=[
        ("preprocesamiento", preprocesamiento),
        ("clasificador", LogisticRegression(
            C=cfg["C"],
            penalty=cfg["penalty"],
            class_weight=cfg["class_weight"],
            solver=cfg["solver"],
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ])

    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    prob = modelo.predict_proba(X_test)[:, 1]

    metricas = {
        "Modelo": cfg["nombre"],
        "C": cfg["C"],
        "class_weight": cfg["class_weight"],
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    }
    resultados.append(metricas)
    modelos[cfg["nombre"]] = modelo

resultados_df = pd.DataFrame(resultados).sort_values(by="F1-score", ascending=False)
display(resultados_df)

In [ ]:
# ============================================================
# 9. Selección del mejor modelo
# ============================================================

mejor_nombre = resultados_df.iloc[0]["Modelo"]
mejor_modelo = modelos[mejor_nombre]

print("Mejor modelo según F1-score:", mejor_nombre)

y_pred = mejor_modelo.predict(X_test)
y_prob = mejor_modelo.predict_proba(X_test)[:, 1]

print("Reporte de clasificación:")
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No segundo", "Segundo/posterior"])
disp.plot(values_format="d")
plt.title(f"Matriz de confusión - {mejor_nombre}")
plt.show()

In [ ]:
# ============================================================
# 10. Gráfica comparativa de métricas
# ============================================================

metricas_graficar = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
resultados_plot = resultados_df.set_index("Modelo")[metricas_graficar]

ax = resultados_plot.plot(kind="bar", figsize=(12, 6))
plt.title("Comparación de modelos de regresión logística")
plt.ylabel("Valor de la métrica")
plt.ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Discusión de resultados

Para este avance se entrenaron varias versiones de regresión logística modificando principalmente el parámetro `C` y el uso de `class_weight`. El parámetro `C` controla la regularización del modelo: valores pequeños aplican mayor regularización y valores grandes permiten un modelo más flexible. También se probó `class_weight="balanced"` para compensar posibles desbalances entre la clase de primer matrimonio y la clase de segundo matrimonio o posterior.

El modelo final se seleccionó usando el F1-score, porque esta métrica equilibra precisión y recall. Esto es importante cuando la variable respuesta puede estar desbalanceada, ya que la exactitud por sí sola puede ser engañosa si una clase aparece con mucha mayor frecuencia que la otra. La matriz de confusión permite observar cuántos casos fueron clasificados correctamente y cuántos errores se cometieron en cada clase.

In [ ]:
# ============================================================
# 11. Guardar resultados para documentación
# ============================================================

resultados_df.to_csv("resultados_logistic_regression.csv", index=False)
print("Archivo generado: resultados_logistic_regression.csv")